In [3]:
import torch
import gc

# Delete the model and processor variables if they exist
try:
    del model
    del processor
except NameError:
    pass

# Force Python's garbage collector to run
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()

In [1]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

Hugging Face cache set to: /workspace/huggingface_cache


In [2]:
import os
import json
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info

# --- Configuration ---
# MiniCPM-V 2.6 (8B)
# TinyChart
# Claude 3.5 Sonnet
# Swap this ID to test Llama or InternVL later
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  

TEST_JSON_PATH = "extracted_chart_specs_test.json"
TEST_IMG_DIR = "./test_images_300"
OUTPUT_FILE = "baseline_qwen_predictoin.json"

def run_baseline_inference():
    print(f"Loading Base Model {MODEL_ID} into VRAM...")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        device_map="auto",
        dtype=torch.bfloat16,
        attn_implementation="sdpa" 
    )
    model.eval()

    with open(TEST_JSON_PATH, 'r', encoding="utf-8") as f:
        ground_truth_data = json.load(f)

    predictions_record = []

    schema_prompt = """Extract the data, topology, and relational structure of this chart into a JSON object matching this exact schema. Omit any keys that have empty lists, 
   empty objects, or null values. Ignore the fields if it is not appropriate for the charttype. 

        {
          "title": "Chart title or null",
          "topo": {
            "type": "line/bar/scatter/pie/area/box/histogram", 
            "sub": "standard/vertical/horizontal/bubble/stacked/unstacked/density/donut"
          },
          "legend": {
            "on": true/false, 
            "title": "Legend title or null"
          },
          "axes": [
            {
              "name": "x_axis or y_axis", 
              "lbl": "Axis label or null", 
              "scl": "linear/log/categorical", 
              "dom": ["min_val", "max_val"]
            }
          ],
          "ser": [
            {
              "name": "Series Name",
              "color": "#hexcode or null",
              "y_ref": "y_axis",
              "data": [["x_val", "y_val"]],
              "trend": {
                "global_trend": "stable/monotonic_increase/monotonic_decrease/fluctuating/peak_then_drop/valley_then_rise",
                "rate_of_change": "stable/slow/moderate/rapid",
                "intervals": [{"x_range": ["x1", "x2"], "trend": "String"}]
              },
              "stats": {
                "max_value": 0.0,
                "max_at_the_ref_point": "x_val",
                "min_value": 0.0,
                "min_at_the_ref_point": "x_val",
                "threshold_coverage": {"above_mean": true, "all_positive": true, "hits_zero": false},
                "crossing_points": [{"series_pair": ["Series A", "Series B"], "ref": "x_val", "value": 0.0}]
              }
            }
          ],
          "rel": [
            {
              "relation_type": "String (e.g., global_maximums_y_axis)", 
              "description": "Natural language description of the relationship.", 
              "ranking": ["Series A", "Series B"]
            }
          ]
        }
        
        Format Rules:
        - For Area charts, 'data' is ["x_val", "y_max", "y_min"].
        - For Box plots, 'data' is ["x_val", "min", "q1", "median", "q3", "max"].
        - For Histograms, 'data' is ["bin_start", "bin_end", "height"].
        - There can be hybrid charts (e.g. scattered points on line/area/bar/box) or charts with outlier points so please also include those scattered points as seperate series with appropriate name.
        - Also include threshhold lines if there are any and count them as a seperate series with name "threshold_line".
        - Return ONLY the raw JSON object. Do not use markdown formatting blocks (```)."""
    processed_ids = set()
    if os.path.exists(OUTPUT_FILE):
        try:
            with open(OUTPUT_FILE, 'r', encoding="utf-8") as f:
                predictions_record = json.load(f)
                processed_ids = {item["id"] for item in predictions_record}
            print(f"🔄 Resuming: Found {len(processed_ids)} existing predictions in {PREDICTIONS_OUTPUT}.")
        except json.JSONDecodeError:
            print(f"⚠️ Warning: {PREDICTIONS_OUTPUT} was corrupted or empty. Starting from scratch.")

    # Filter out already processed items
    remaining_data = [item for item in ground_truth_data if item.get("id") not in processed_ids]
    print(f"📊 Total: {len(ground_truth_data)} | Already Done: {len(processed_ids)} | Remaining: {len(remaining_data)}\n")

    for index, gt_item in enumerate(remaining_data, start=1):
        chart_id = gt_item.get("id")
        ctype = gt_item.get("chart_type")
        img_path = os.path.join(TEST_IMG_DIR, f"{chart_id}.png")
        
        print(f"Zero-Shot Eval [{index}/{len(ground_truth_data)}] - {ctype} (ID: {chart_id})")
        if not os.path.exists(img_path): continue

        messages = [
            {"role": "system", "content": "You output strictly valid JSON with no markdown formatting or extra text."},
            {"role": "user", "content": [
                {"type": "image", "image": img_path, "max_pixels": 768 * 768},
                {"type": "text", "text": schema_prompt}
            ]}
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs, 
                max_new_tokens=4096,
                temperature=0.1,  
                do_sample=False
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        prediction_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]
        
        # Clean up Markdown backticks if the model ignored the system prompt
        if prediction_text.startswith("```json"):
            prediction_text = prediction_text.replace("```json", "").replace("```", "").strip()

        predictions_record.append({
            "id": chart_id,
            "chart_type": ctype,
            "predicted_text": prediction_text
        })

        # --- THE SAVE-AFTER-EACH-INFERENCE FIX ---
        # We overwrite the file every time. Since it's JSON, this is safest for data integrity.
        with open(OUTPUT_FILE, 'w', encoding="utf-8") as f:
            json.dump(predictions_record, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Baseline predictions saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    run_baseline_inference()

Loading Base Model Qwen/Qwen2.5-VL-7B-Instruct into VRAM...


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

📊 Total: 258 | Already Done: 0 | Remaining: 258

Zero-Shot Eval [1/258] - Histogram (ID: 131740)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Zero-Shot Eval [2/258] - Line Chart (ID: 149999)
Zero-Shot Eval [3/258] - Line Chart (ID: 153147)
Zero-Shot Eval [4/258] - Line Chart (ID: 149267)
Zero-Shot Eval [5/258] - Line Chart (ID: 107744)
Zero-Shot Eval [6/258] - Line Chart (ID: 90311)
Zero-Shot Eval [7/258] - Area Chart (ID: 55210)
Zero-Shot Eval [8/258] - Pie Chart (ID: 71634)
Zero-Shot Eval [9/258] - Area Chart (ID: 122595)
Zero-Shot Eval [10/258] - Scatter Plot (ID: 50003)
Zero-Shot Eval [11/258] - Scatter Plot (ID: 118774)
Zero-Shot Eval [12/258] - Box Plot (ID: 50419)
Zero-Shot Eval [13/258] - Bar Chart (ID: 147731)
Zero-Shot Eval [14/258] - Bar Chart (ID: 154422)
Zero-Shot Eval [15/258] - Pie Chart (ID: 66936)
Zero-Shot Eval [16/258] - Line Chart (ID: 56537)
Zero-Shot Eval [17/258] - Scatter Plot (ID: 59868)
Zero-Shot Eval [18/258] - Pie Chart (ID: 155348)
Zero-Shot Eval [19/258] - Box Plot (ID: 105282)
Zero-Shot Eval [20/258] - Line Chart (ID: 65440)
Zero-Shot Eval [21/258] - Line Chart (ID: 150369)
Zero-Shot Eval [22/2